# Notebook 06 – BiLSTM Autoencoder for Stress Pattern Extraction

**Objective:** Learn the compressed 32-dimensional stress embeddings from 6-month temporal payment behavior sequences using a Bidirectional LSTM autoencoder.

**Input:** `sbdr_final_dataset.csv` (Phase 1 output)  
**Output:** `06_with_stress_vectors.csv` with 32 stress dimensions + anomaly flags

---

In [ ]:
# Cell 1: Imports
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.backends.mps.is_available():
    print("MPS (Apple Silicon) available")

In [ ]:
# Cell 2: Configuration

CONFIG = {
    # ----- File Paths -----
    'input_path': 'sbdr_final_dataset.csv',
    'output_path': '06_with_stress_vectors.csv',
    'model_dir': 'models/bilstm',
    'manifest_path': 'bilstm_manifest.json',

    # ----- Column Names (from Phase 1 CSV) -----
    # Payment status: 6 months (NOTE: UCI uses PAY_0, PAY_2-6, no PAY_1)
    'pay_status_cols': ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6'],

    # Bill amounts: 6 months (BILL_AMT1 = most recent)
    'bill_amt_cols': ['BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6'],

    # Payment amounts: 6 months (PAY_AMT1 = most recent)
    'pay_amt_cols': ['PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6'],

    # Utilization ratios: 6 months (from Phase 1 feature engineering)
    'util_ratio_cols': ['util_ratio_1', 'util_ratio_2', 'util_ratio_3', 'util_ratio_4', 'util_ratio_5', 'util_ratio_6'],

    # Pay ratios: 6 months (from Phase 1 feature engineering)
    'pay_ratio_cols': ['pay_ratio_1', 'pay_ratio_2', 'pay_ratio_3', 'pay_ratio_4', 'pay_ratio_5', 'pay_ratio_6'],

    # Target column
    'target_col': 'default payment next month',

    # ----- Model Architecture -----
    'seq_len': 6,           # 6 months of history
    'n_features': 5,        # Features per timestep: pay_status, bill_amt, pay_amt, util_ratio, pay_ratio
    'hidden_size': 64,      # LSTM hidden dimension
    'num_layers': 2,        # LSTM layers
    'embed_dim': 32,        # Bottleneck / stress vector dimension
    'dropout': 0.2,

    # ----- Training -----
    'batch_size': 256,
    'epochs': 100,
    'lr': 1e-3,
    'patience': 10,         # Early stopping patience
    'grad_clip': 1.0,

    # ----- Anomaly Detection -----
    'anomaly_percentile': 95,  # Reconstruction error threshold percentile

    # ----- Random Seed -----
    'seed': 42
}

# Set seeds for reproducibility
np.random.seed(CONFIG['seed'])
torch.manual_seed(CONFIG['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG['seed'])

# Device selection
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')
print(f"Using device: {DEVICE}")

# Create model directory
Path(CONFIG['model_dir']).mkdir(parents=True, exist_ok=True)
print("Configuration loaded.")

In [ ]:
# Cell 3: Load Data and Build Sequences

def build_sequences(df, config):
    """
    Build 3D tensor of shape (N, seq_len, n_features) from wide DataFrame.

    Features per timestep:
    - Payment status (categorical, kept as-is)
    - Bill amount (log1p scaled)
    - Payment amount (log1p scaled)
    - Utilization ratio (already 0-1 range, clipped)
    - Pay ratio (clipped to reasonable range)
    """
    n_samples = len(df)
    seq_len = config['seq_len']
    n_features = config['n_features']

    # Initialize tensor
    sequences = np.zeros((n_samples, seq_len, n_features), dtype=np.float32)

    # Feature 0: Payment status (already integer -2 to 8 range)
    pay_status = df[config['pay_status_cols']].values.astype(np.float32)
    # Normalize to ~[-1, 1] range: divide by 4 (since range is roughly -2 to 8)
    sequences[:, :, 0] = pay_status / 4.0

    # Feature 1: Bill amounts (log1p transform)
    bill_amt = df[config['bill_amt_cols']].values.astype(np.float32)
    bill_amt = np.clip(bill_amt, 0, None)  # Ensure non-negative
    bill_amt_log = np.log1p(bill_amt)
    # Standardize
    bill_mean, bill_std = bill_amt_log.mean(), bill_amt_log.std() + 1e-8
    sequences[:, :, 1] = (bill_amt_log - bill_mean) / bill_std

    # Feature 2: Payment amounts (log1p transform)
    pay_amt = df[config['pay_amt_cols']].values.astype(np.float32)
    pay_amt = np.clip(pay_amt, 0, None)
    pay_amt_log = np.log1p(pay_amt)
    pay_mean, pay_std = pay_amt_log.mean(), pay_amt_log.std() + 1e-8
    sequences[:, :, 2] = (pay_amt_log - pay_mean) / pay_std

    # Feature 3: Utilization ratio (clip to [0, 2] to handle edge cases)
    util_ratio = df[config['util_ratio_cols']].values.astype(np.float32)
    util_ratio = np.clip(util_ratio, 0, 2)
    # Center around 0.5 (typical utilization)
    sequences[:, :, 3] = (util_ratio - 0.5) * 2  # Maps [0,1] to [-1,1]

    # Feature 4: Pay ratio (clip extreme values)
    pay_ratio = df[config['pay_ratio_cols']].values.astype(np.float32)
    pay_ratio = np.clip(pay_ratio, 0, 5)  # Cap at 5x payment
    # Log transform to handle skew
    pay_ratio_log = np.log1p(pay_ratio)
    pr_mean, pr_std = pay_ratio_log.mean(), pay_ratio_log.std() + 1e-8
    sequences[:, :, 4] = (pay_ratio_log - pr_mean) / pr_std

    # Store normalization params for inference
    norm_params = {
        'bill_mean': float(bill_mean), 'bill_std': float(bill_std),
        'pay_mean': float(pay_mean), 'pay_std': float(pay_std),
        'pay_ratio_mean': float(pr_mean), 'pay_ratio_std': float(pr_std)
    }

    return sequences, norm_params

# Load data
print("Loading data...")
df = pd.read_csv(CONFIG['input_path'])
print(f"Loaded {len(df):,} rows, {len(df.columns)} columns")

# Verify required columns exist
required_cols = (CONFIG['pay_status_cols'] + CONFIG['bill_amt_cols'] +
                 CONFIG['pay_amt_cols'] + CONFIG['util_ratio_cols'] +
                 CONFIG['pay_ratio_cols'])
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")
print(f"All {len(required_cols)} required temporal columns found")

# Build sequences
print("\nBuilding sequences...")
sequences, norm_params = build_sequences(df, CONFIG)
print(f"Sequence tensor shape: {sequences.shape}")
print(f"  - {sequences.shape[0]:,} samples")
print(f"  - {sequences.shape[1]} timesteps (months)")
print(f"  - {sequences.shape[2]} features per timestep")

In [ ]:
# Cell 4: Creating DataLoaders

# Convert to PyTorch tensors
X_tensor = torch.FloatTensor(sequences)

# Train/val split (80/20)
n_train = int(0.8 * len(X_tensor))
indices = torch.randperm(len(X_tensor))
train_idx, val_idx = indices[:n_train], indices[n_train:]

X_train = X_tensor[train_idx]
X_val = X_tensor[val_idx]

# Creating datasets and loaders
train_dataset = TensorDataset(X_train, X_train)  # Autoencoder: input = target
val_dataset = TensorDataset(X_val, X_val)

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False)

print(f"Training samples: {len(X_train):,}")
print(f"Validation samples: {len(X_val):,}")
print(f"Batches per epoch: {len(train_loader)}")

In [ ]:
# Cell 5: BiLSTM Encoder

class BiLSTMEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, embed_dim, num_layers=2, dropout=0.2):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.bottleneck = nn.Sequential(
            nn.Linear(hidden_dim * 2, embed_dim),
            nn.Tanh()
        )

    def forward(self, x):
        lstm_out, (h_n, c_n) = self.lstm(x)
        h_forward = h_n[-2, :, :]
        h_backward = h_n[-1, :, :]
        h_combined = torch.cat([h_forward, h_backward], dim=1)
        embedding = self.bottleneck(h_combined)
        return embedding, lstm_out

In [ ]:
# Cell 6: LSTM Decoder and Full Autoencoder

class LSTMDecoder(nn.Module):
    def __init__(self, embed_dim, hidden_dim, output_dim, seq_len, num_layers=2, dropout=0.2):
        super().__init__()
        self.seq_len = seq_len
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.embed_to_hidden = nn.Linear(embed_dim, hidden_dim * num_layers)
        self.embed_to_cell = nn.Linear(embed_dim, hidden_dim * num_layers)

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.output_layer = nn.Linear(hidden_dim, output_dim)

    def forward(self, embedding):
        batch_size = embedding.size(0)

        h_0 = self.embed_to_hidden(embedding)
        c_0 = self.embed_to_cell(embedding)

        h_0 = h_0.view(batch_size, self.num_layers, self.hidden_dim).permute(1, 0, 2).contiguous()
        c_0 = c_0.view(batch_size, self.num_layers, self.hidden_dim).permute(1, 0, 2).contiguous()

        decoder_input = embedding.unsqueeze(1).repeat(1, self.seq_len, 1)
        lstm_out, _ = self.lstm(decoder_input, (h_0, c_0))
        reconstruction = self.output_layer(lstm_out)

        return reconstruction


class BiLSTMAutoencoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.encoder = BiLSTMEncoder(
            input_dim=config['n_features'],
            hidden_dim=config['hidden_size'],
            embed_dim=config['embed_dim'],
            num_layers=config['num_layers'],
            dropout=config['dropout']
        )
        self.decoder = LSTMDecoder(
            embed_dim=config['embed_dim'],
            hidden_dim=config['hidden_size'],
            output_dim=config['n_features'],
            seq_len=config['seq_len'],
            num_layers=config['num_layers'],
            dropout=config['dropout']
        )

    def forward(self, x):
        embedding, encoder_outputs = self.encoder(x)
        reconstruction = self.decoder(embedding)
        return reconstruction, embedding

    def encode(self, x):
        embedding, _ = self.encoder(x)
        return embedding


# Initialize model
model = BiLSTMAutoencoder(CONFIG).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model initialized with {n_params:,} trainable parameters")
print(f"Encoder: BiLSTM({CONFIG['n_features']} -> {CONFIG['hidden_size']}x2) -> Bottleneck({CONFIG['embed_dim']})")
print(f"Decoder: LSTM({CONFIG['embed_dim']} -> {CONFIG['hidden_size']}) -> Output({CONFIG['n_features']})")

In [ ]:
# Cell 7: Training Loop

def train_epoch(model, loader, optimizer, criterion, device, grad_clip):
    model.train()
    total_loss = 0
    for batch_x, batch_y in loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        reconstruction, _ = model(batch_x)
        loss = criterion(reconstruction, batch_y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        total_loss += loss.item() * len(batch_x)
    return total_loss / len(loader.dataset)


def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            reconstruction, _ = model(batch_x)
            loss = criterion(reconstruction, batch_y)
            total_loss += loss.item() * len(batch_x)
    return total_loss / len(loader.dataset)


criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['lr'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)

history = {'train_loss': [], 'val_loss': []}
best_val_loss = float('inf')
patience_counter = 0
best_model_path = Path(CONFIG['model_dir']) / 'best_model.pt'

print("Starting training...\n")
for epoch in range(CONFIG['epochs']):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, DEVICE, CONFIG['grad_clip'])
    val_loss = validate(model, val_loader, criterion, DEVICE)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), best_model_path)
    else:
        patience_counter += 1

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{CONFIG['epochs']} | Train: {train_loss:.6f} | Val: {val_loss:.6f} | Best: {best_val_loss:.6f} | Patience: {patience_counter}/{CONFIG['patience']}")

    if patience_counter >= CONFIG['patience']:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

print(f"\nTraining complete. Best validation loss: {best_val_loss:.6f}")

In [ ]:
# Cell 8: Training Visualization

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(history['train_loss'], label='Train Loss')
ax.plot(history['val_loss'], label='Validation Loss')
ax.axhline(y=best_val_loss, color='r', linestyle='--', alpha=0.5, label=f'Best: {best_val_loss:.6f}')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('BiLSTM Autoencoder Training')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(Path(CONFIG['model_dir']) / 'training_loss.png', dpi=150)
plt.show()

In [ ]:
# Cell 9: Extract Embeddings

model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
model.eval()

all_embeddings = []
all_recon_errors = []

full_loader = DataLoader(TensorDataset(X_tensor, X_tensor), batch_size=CONFIG['batch_size'], shuffle=False)

print("Extracting stress embeddings...")
with torch.no_grad():
    for batch_x, batch_y in full_loader:
        batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
        reconstruction, embedding = model(batch_x)
        all_embeddings.append(embedding.cpu().numpy())
        recon_error = ((reconstruction - batch_y) ** 2).mean(dim=[1, 2])
        all_recon_errors.append(recon_error.cpu().numpy())

embeddings = np.vstack(all_embeddings)
recon_errors = np.concatenate(all_recon_errors)

print(f"Extracted {embeddings.shape[0]:,} embeddings of dimension {embeddings.shape[1]}")
print(f"Reconstruction error: mean={recon_errors.mean():.6f}, std={recon_errors.std():.6f}")

In [ ]:
# Cell 10: Anomaly Detection

anomaly_threshold = np.percentile(recon_errors, CONFIG['anomaly_percentile'])
anomaly_flags = (recon_errors > anomaly_threshold).astype(int)

print(f"Anomaly threshold ({CONFIG['anomaly_percentile']}th %ile): {anomaly_threshold:.6f}")
print(f"Anomalies detected: {anomaly_flags.sum():,} ({100*anomaly_flags.mean():.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(recon_errors, bins=50, alpha=0.7, edgecolor='black')
axes[0].axvline(anomaly_threshold, color='r', linestyle='--', linewidth=2, label=f'Threshold: {anomaly_threshold:.4f}')
axes[0].set_xlabel('Reconstruction Error')
axes[0].set_ylabel('Count')
axes[0].set_title('Reconstruction Error Distribution')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

if CONFIG['target_col'] in df.columns:
    defaulters = df[CONFIG['target_col']].values == 1
    axes[1].hist(recon_errors[~defaulters], bins=50, alpha=0.6, label='Non-default', density=True)
    axes[1].hist(recon_errors[defaulters], bins=50, alpha=0.6, label='Default', density=True)
    axes[1].axvline(anomaly_threshold, color='r', linestyle='--', linewidth=2)
    axes[1].set_xlabel('Reconstruction Error')
    axes[1].set_ylabel('Density')
    axes[1].set_title('Error by Default Status')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    print(f"\nAnomaly rate - Defaulters: {100*anomaly_flags[defaulters].mean():.1f}%, Non-defaulters: {100*anomaly_flags[~defaulters].mean():.1f}%")

plt.tight_layout()
plt.savefig(Path(CONFIG['model_dir']) / 'recon_error_dist.png', dpi=150)
plt.show()

In [ ]:
# Cell 11: Save Output CSV

stress_cols = [f'stress_dim_{i:02d}' for i in range(CONFIG['embed_dim'])]
embeddings_df = pd.DataFrame(embeddings, columns=stress_cols, index=df.index)
embeddings_df['bilstm_recon_error'] = recon_errors
embeddings_df['bilstm_anomaly_flag'] = anomaly_flags

df_output = pd.concat([df, embeddings_df], axis=1)
df_output.to_csv(CONFIG['output_path'], index=False)

print(f"Output saved to: {CONFIG['output_path']}")
print(f"Total columns: {len(df_output.columns)} (added {len(stress_cols) + 2} new)")
print(f"  - 32 stress dimensions: {stress_cols[0]} to {stress_cols[-1]}")
print(f"  - bilstm_recon_error")
print(f"  - bilstm_anomaly_flag")

In [ ]:
# Cell 12: Save Manifest

manifest = {
    'notebook': '06_bilstm_stress_patterns',
    'output_file': CONFIG['output_path'],
    'model_checkpoint': str(best_model_path),
    'stress_embedding_cols': stress_cols,
    'metadata_cols': ['bilstm_recon_error', 'bilstm_anomaly_flag'],
    'model_config': {k: CONFIG[k] for k in ['seq_len', 'n_features', 'hidden_size', 'num_layers', 'embed_dim', 'dropout']},
    'input_features': {k: CONFIG[k] for k in ['pay_status_cols', 'bill_amt_cols', 'pay_amt_cols', 'util_ratio_cols', 'pay_ratio_cols']},
    'normalization_params': norm_params,
    'anomaly_detection': {
        'threshold_percentile': CONFIG['anomaly_percentile'],
        'threshold_value': float(anomaly_threshold),
        'n_anomalies': int(anomaly_flags.sum()),
        'anomaly_rate': float(anomaly_flags.mean())
    },
    'training_summary': {
        'best_val_loss': float(best_val_loss),
        'epochs_trained': len(history['train_loss'])
    }
}

with open(CONFIG['manifest_path'], 'w') as f:
    json.dump(manifest, f, indent=2)

print(f"Manifest saved to: {CONFIG['manifest_path']}")
print("\n" + "="*50)
print("NOTEBOOK 06 COMPLETE")
print("="*50)

In [ ]:
# Cell 13: Sanity Check

print("SANITY CHECK")
print("="*50)

df_check = pd.read_csv(CONFIG['output_path'])
print(f"1. Output readable: OK - Shape {df_check.shape}")
print(f"2. Stress columns present: {'OK' if all(c in df_check.columns for c in stress_cols) else 'FAIL'}")
print(f"3. No NaN in stress cols: {'OK' if df_check[stress_cols].isna().sum().sum() == 0 else 'FAIL'}")
print(f"4. Anomaly flag binary: {'OK' if set(df_check['bilstm_anomaly_flag'].unique()) <= {0, 1} else 'FAIL'}")

if CONFIG['target_col'] in df_check.columns:
    corr = df_check['bilstm_recon_error'].corr(df_check[CONFIG['target_col']])
    print(f"5. Recon error vs default correlation: {corr:.4f}")

print("\n" + "="*50)
print("Ready for Notebook 07 (XGBoost)!")
print("="*50)